# Notebook: 03_maintain_gold
### Descripción: OPTIMIZE + VACUUM de las tablas Gold
Schedule: Domingos 3am ART (06:00 UTC) — después de maintain_silver


### 🥇 Maintenance — Gold Layer
OPTIMIZE con ZORDER en fact_ventas para acelerar queries del dashboard.
VACUUM en todas las tablas Gold para liberar storage.

In [0]:
%sql
-- Estado antes
DESCRIBE DETAIL iniciacion_deportiva.gold.fact_ventas;

In [0]:
%sql
DESCRIBE DETAIL iniciacion_deportiva.gold.dim_producto;

In [0]:
%sql
DESCRIBE DETAIL iniciacion_deportiva.gold.dim_tiempo;

In [0]:
%sql
-- OPTIMIZE fact_ventas
-- ZORDER BY tiempo_key: todas las queries del dashboard filtran por fecha.
-- Esto activa el data skipping de Delta — Databricks lee solo los archivos
-- que contienen el rango de fechas pedido, ignorando el resto.

OPTIMIZE iniciacion_deportiva.gold.fact_ventas
ZORDER BY (tiempo_key);

In [0]:
%sql
-- OPTIMIZE dim_producto
-- ZORDER BY id_producto: los JOINs desde fact_ventas siempre van por id_producto.
OPTIMIZE iniciacion_deportiva.gold.dim_producto
ZORDER BY (id_producto);

In [0]:
%sql
-- OPTIMIZE dim_tiempo
-- Sin ZORDER: tabla chica (~365 filas), se lee completa en cada join.
OPTIMIZE iniciacion_deportiva.gold.dim_tiempo;

In [0]:
%sql
-- VACUUM fact_ventas
-- Gold es la capa más consultada — limpiar historial viejo mejora performance.
-- VACUUM
VACUUM iniciacion_deportiva.gold.fact_ventas RETAIN 168 HOURS;

In [0]:
%sql
-- VACUUM dim_producto
-- SCD Type 2 genera versiones históricas — VACUUM limpia las obsoletas.
VACUUM iniciacion_deportiva.gold.dim_producto RETAIN 168 HOURS;

In [0]:
%sql
-- VACUUM dim_tiempo
VACUUM iniciacion_deportiva.gold.dim_tiempo RETAIN 168 HOURS;

In [0]:
%sql
-- Estado DESPUÉS
DESCRIBE DETAIL iniciacion_deportiva.gold.fact_ventas;

In [0]:
%sql
DESCRIBE DETAIL iniciacion_deportiva.gold.dim_producto;